# Final Chart



This is just un example of the final chart. It will use a pre-trained model for Labrea station `13870000`

In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from cn3s import CN3S, CN3SOptimizer, CN3SParams
from cn3s.objectives import Objectives
from hydrography import Hydrography, Stations


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
STATION = "13870000"
FREQ = "M"
DATA = Path("/data/CN3S")


## Test CN3S

In [5]:

q = pd.read_parquet("/data/CN3S/stations/13870000/discharge/monthly.parquet")
prec = pd.read_parquet("/data/CN3S/stations/13870000/rain/monthly.parquet")

In [249]:
params = CN3SParams(
    name=f"LABREA ({STATION})",
    area=226439,
    warmup_steps=10,
    r0=0,
    cn_i=25,
    alfa=0.2,
    beta=0.0026,
    k0=0.4,  # decay k0** i
    k1=0.53,
    k2=0.6,
    act=1,
)

model = CN3S(params, freq="M")

model.run(prec["prec"])
model.evaluate(q["Vazao"])

Running CN3S model:   0%|          | 0/309 [00:00<?, ?step/s]

Aligned 306 time steps between model results and observations.
From 2000-12-01 00:00:00 to 2026-05-01 00:00:00.


0.7736859180483708

In [250]:
model.plot()

In [251]:
model.params.warmup_steps

10

In [177]:
model.results.iloc[-2]["past_prec"]

[55.87694474248927,
 46.94175831545064,
 170.59951716738198,
 222.55733637339057,
 151.22337714592274,
 312.41868964592277,
 263.39615745708153,
 136.17318267167383,
 102.953125,
 97.07145252145922]

In [182]:
model.results.iloc[-1]["past_prec"][:5]

[16.820245439914164,
 55.87694474248927,
 46.94175831545064,
 170.59951716738198,
 222.55733637339057]

In [183]:
ap = sum((0.47**i) * float(p) for i, p in enumerate(model.results.iloc[-1]["past_prec"][:5]))
vj = 1.0 + 0.0026 * ap
vj

1.213262622582762

In [232]:
model.results.tail(30)

,prec,past_prec,vj,cnv,s,q_up,r1,q_low,r,q_mm,q_m3s,obs_q_m3s
2024-04-01,131.327488,"[225.3260293723176, 209.2763546137339, 190.272...",1.990537,43.057792,335.904838,10.28562,114.628122,74.508279,40.119843,84.793899,7407.656512,10612.666000
2024-05-01,96.306431,"[131.32748792918454, 225.3260293723176, 209.27...",1.806847,39.793968,384.287689,0.936897,90.665696,58.932702,31.732994,59.869599,5230.251596,6776.818710
2024-06-01,46.651757,"[96.30643106223177, 131.32748792918454, 225.32...",1.629568,36.585021,440.273214,0.0,56.458425,36.697976,19.760449,36.697976,3205.961811,2633.753333
2024-07-01,8.231894,"[46.651756974248926, 96.30643106223177, 131.32...",1.417181,32.653277,523.869871,0.0,24.123352,15.680179,8.443173,15.680179,1369.831815,1283.507419
2024-08-01,9.426905,"[8.231893776824034, 46.651756974248926, 96.306...",1.217454,28.85457,626.276512,0.0,13.439433,8.735631,4.703801,8.735631,763.151084,797.093226
2024-09-01,6.794394,"[9.426904506437769, 8.231893776824034, 46.6517...",1.126637,27.089621,683.628469,0.0,8.30483,5.39814,2.906691,5.39814,471.585391,579.551000
2024-10-01,31.526723,"[6.794393776824034, 9.426904506437769, 8.23189...",1.077077,26.115387,718.606694,0.0,19.615854,12.750305,6.865549,12.750305,1113.8759,572.877895
2024-11-01,92.370809,"[31.52672344420601, 6.794393776824034, 9.42690...",1.118068,26.921757,689.47481,0.0,55.822078,36.28435,19.537727,36.28435,3169.827168,1096.886000
2024-12-01,130.26311,"[92.3708087446352, 31.52672344420601, 6.794393...",1.295396,30.349815,582.9079,0.313757,88.410884,57.467075,30.943809,57.780832,5047.775393,3283.071935
2025-01-01,202.771929,"[130.2631102467811, 92.3708087446352, 31.52672...",1.477234,33.775452,498.025473,17.703812,129.029911,83.869442,45.160469,101.573254,8873.51318,7353.027742


### Predict next discharges


In [22]:
prec["prec"].tail()

2026-04-01    170.599517
2026-05-01     46.941758
2026-06-01     55.876945
2026-07-01     16.820245
2026-08-01     46.093214
Name: prec, dtype: float64

In [76]:
avg_prec = pd.Series([56.73, 112.4, 153.5, 214.1], index=pd.date_range(start="2026-09-01", freq="MS", periods=4))

# prepend the last observed precipitation values
avg_prec = pd.concat([prec["prec"], avg_prec])
avg_prec

2000-01-01    186.007980
2000-02-01    158.437332
2000-03-01    157.134690
2000-04-01    149.412554
2000-05-01     64.597170
                 ...    
2026-08-01     46.093214
2026-09-01     56.730000
2026-10-01    112.400000
2026-11-01    153.500000
2026-12-01    214.100000
Length: 324, dtype: float64

In [77]:
fcst_prec = pd.Series([56.73, 77.64, 103.9, 158.6], index=pd.date_range(start="2026-09-01", freq="MS", periods=4))

# prepend the last observed precipitation values
fcst_prec = pd.concat([prec["prec"], fcst_prec])

In [224]:
# predict next discharges for observed average prec
model.run(avg_prec)
fcst_avg_prec = model.results["q_m3s"]
fcst_avg_prec.index = fcst_avg_prec.index + pd.DateOffset(days=14)
fcst_avg_prec

Running CN3S model:   0%|          | 0/320 [00:00<?, ?step/s]

2000-06-15    7702.109991
2000-07-15    4444.309618
2000-08-15    2701.264739
2000-09-15    1734.938943
2000-10-15    2163.780556
                 ...     
2026-10-15     1943.51668
2026-11-15    2387.563863
2026-12-15    4218.411885
2027-01-15    6266.342146
2027-02-15    9827.282411
Name: q_m3s, Length: 321, dtype: object

In [225]:
# predict next discharges for forecasted precipitation
model.run(fcst_prec)
fcst_fcst_prec = model.results["q_m3s"]
fcst_fcst_prec.index = fcst_fcst_prec.index + pd.DateOffset(days=14)
fcst_fcst_prec

Running CN3S model:   0%|          | 0/320 [00:00<?, ?step/s]

2000-06-15    7702.109991
2000-07-15    4444.309618
2000-08-15    2701.264739
2000-09-15    1734.938943
2000-10-15    2163.780556
                 ...     
2026-10-15     1943.51668
2026-11-15    2387.563863
2026-12-15    3172.282925
2027-01-15    4237.249513
2027-02-15     6464.91275
Name: q_m3s, Length: 321, dtype: object

## Construct final Chart 

In [18]:
from hydro.station import Station
from utils.sql_connector import SqlConnector

In [20]:
conn = SqlConnector()

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code FGJV64UM3 to authenticate.


In [21]:
station = Station(STATION, connector=conn)

In [35]:
qd = station.get_series("hidro/telemetria", "vazao")
qd

,EstacaoCodigo,NivelConsistencia,MediaDiaria,vazao_data,val,ano,mes,dia,Sistema,HORESTACAO,ORIGEM
Data,,,,,,,,,,,
1967-07-01,13870000.0,2,1.0,1967-07-01,2629.860107,1967.0,7.0,1.0,Hidro,NaN,NaN
1967-07-02,13870000.0,2,1.0,1967-07-02,2546.629883,1967.0,7.0,2.0,Hidro,NaN,NaN
1967-07-03,13870000.0,2,1.0,1967-07-03,2455.729980,1967.0,7.0,3.0,Hidro,NaN,NaN
1967-07-04,13870000.0,2,1.0,1967-07-04,2384.199951,1967.0,7.0,4.0,Hidro,NaN,NaN
1967-07-05,13870000.0,2,1.0,1967-07-05,2313.729980,1967.0,7.0,5.0,Hidro,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-09-27,NaN,1,NaN,NaN,1150.882000,NaN,NaN,NaN,Telemetria,71564480.0,RHN
2026-09-28,NaN,1,NaN,NaN,1130.155000,NaN,NaN,NaN,Telemetria,71564480.0,RHN
2026-09-29,NaN,1,NaN,NaN,1126.925000,NaN,NaN,NaN,Telemetria,71564480.0,RHN


In [36]:
stats = pd.DataFrame(
    {
        "median": qd.groupby([qd.index.month, qd.index.day])["val"].median(),
        "min": qd.groupby([qd.index.month, qd.index.day])["val"].min(),
        "max": qd.groupby([qd.index.month, qd.index.day])["val"].max(),
    },
)

stats = stats.drop((2, 29))
stats.index = pd.date_range(start="2026-01-01", end="2026-12-31")

s2025 = qd.loc["2025"]
s2026 = qd.loc["2026"]
s2024 = qd.loc["2024"]

monthly_mean = qd.groupby(qd.index.month)["val"].median()

In [38]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=stats.index, y=stats["median"], mode="lines", name="Median", opacity=0.1, line={"color": "blue", "dash": "dash"})
fig.add_scatter(x=stats.index, y=stats["min"], mode="lines", name="Min", opacity=0.1, line={"color": "grey", "width": 0})
fig.add_scatter(x=stats.index, y=stats["max"], mode="lines", name="Max", opacity=0.01, line={"color": "grey", "width": 0}, fill="tonexty", fillcolor="rgba(128, 128, 128, 0.10)")
fig.add_scatter(x=stats.index, y=s2025["val"], mode="lines", name="2025", line={"color": "blue", "width": 1})
fig.add_scatter(x=stats.index, y=s2026["val"], mode="lines", name="2026", line={"color": "black"})
fig.add_scatter(x=stats.index, y=s2024["val"], mode="lines", name="2024", line={"color": "green", "width": 1})

monthly_mean = qd.groupby(qd.index.month)["val"].median()

month_edges = pd.date_range("2026-01-01", "2027-01-01", freq="MS")
month_values = monthly_mean.reindex(month_edges[:-1].month).to_numpy()

monthly_line_x = []
monthly_line_y = []
for month_start, month_end, monthly_value in zip(
    month_edges[:-1], month_edges[1:], month_values
):
    monthly_line_x.extend([month_start, month_end, None])
    monthly_line_y.extend([monthly_value, monthly_value, None])

fig.add_scatter(
    x=monthly_line_x,
    y=monthly_line_y,
    mode="lines",
    name="Monthly mean",
    line={"color": "orange", "width": 1, "dash": "dot"},
)

# ...existing code...
fig.update_layout(
    width=1000,
    height=500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showline": True, "linecolor": "black", "mirror": True},
    yaxis={"showline": True, "linecolor": "black", "mirror": True},
)
fig.show()

In [226]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=stats.index, y=stats["median"], mode="lines", name="Median", opacity=0.1, line={"color": "blue", "dash": "dash"})
fig.add_scatter(x=stats.index, y=stats["min"], mode="lines", name="Min", opacity=0.1, line={"color": "grey", "width": 0})
fig.add_scatter(x=stats.index, y=stats["max"], mode="lines", name="Max", opacity=0.01, line={"color": "grey", "width": 0}, fill="tonexty", fillcolor="rgba(128, 128, 128, 0.10)")
fig.add_scatter(x=stats.index, y=s2025["val"], mode="lines", name="2025", line={"color": "blue", "width": 1})
fig.add_scatter(x=stats.index, y=s2026["val"], mode="lines", name="2026", line={"color": "black"})
fig.add_scatter(x=stats.index, y=s2024["val"], mode="lines", name="2024", line={"color": "green", "width": 1})

monthly_mean = qd.groupby(qd.index.month)["val"].median()
monthly_mean = pd.Series(monthly_mean.values, index=pd.date_range("2026-01-01", freq="MS", periods=12))
monthly_mean.index = monthly_mean.index + pd.DateOffset(days=14)

fig.add_scatter(
    x=monthly_mean.index,
    y=monthly_mean.values,
    mode="markers",
    name="Média Mensal",
    marker={"color": "orange", "size": 6, "symbol": "circle"},
)

fig.add_scatter(
    x=fcst_avg_prec.loc["2026-10": "2026-12"].index,
    y=fcst_avg_prec.loc["2026-10": "2026-12"].values,
    mode="markers",
    name="Previsão (chuva média)",
    marker={"color": "green", "size": 10, "symbol": "cross"},
)

fig.add_scatter(
    x=fcst_fcst_prec.loc["2026-10": "2026-12"].index,
    y=fcst_fcst_prec.loc["2026-10": "2026-12"].values,
    mode="markers",
    name="Previsão (chuva prevista)",
    marker={"color": "red", "size": 10, "symbol": "x"},
)

fig.add_scatter(
    x=monthly_line_x,
    y=monthly_line_y,
    mode="lines",
    name="Monthly mean",
    line={"color": "orange", "width": 1, "dash": "dot"},
)

# ...existing code...
fig.update_layout(
    width=1200,
    height=600,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showline": True, "linecolor": "black", "mirror": True},
    yaxis={"showline": True, "linecolor": "black", "mirror": True, "type": "linear"},
)
fig.show()

In [55]:
monthly_mean = pd.Series(monthly_mean.values, index=pd.date_range("2026-01-01", freq="MS", periods=12))
monthly_mean.index = monthly_mean.index + pd.DateOffset(days=14)

In [127]:
fcst_avg_prec.loc["2026"]

2026-01-15    10607.520359
2026-02-15    13604.693327
2026-03-15      8609.47026
2026-04-15     9880.868771
2026-05-15     8394.436972
2026-06-15     3728.703499
2026-07-15     2872.552417
2026-08-15     1302.236069
2026-09-15     1926.186802
2026-10-15     2451.546344
2026-11-15     4516.076432
2026-12-15     6459.442964
Name: q_m3s, dtype: object

In [112]:
s2024

,EstacaoCodigo,NivelConsistencia,MediaDiaria,vazao_data,val,ano,mes,dia,Sistema,HORESTACAO,ORIGEM
Data,,,,,,,,,,,
2024-01-01,13870000.0,1,1.0,2024-01-01,4232.889160,2024.0,1.0,1.0,Hidro,NaN,NaN
2024-01-02,13870000.0,1,1.0,2024-01-02,4301.686523,2024.0,1.0,2.0,Hidro,NaN,NaN
2024-01-03,13870000.0,1,1.0,2024-01-03,4389.380859,2024.0,1.0,3.0,Hidro,NaN,NaN
2024-01-04,13870000.0,1,1.0,2024-01-04,4484.299805,2024.0,1.0,4.0,Hidro,NaN,NaN
2024-01-05,13870000.0,1,1.0,2024-01-05,4599.188965,2024.0,1.0,5.0,Hidro,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,13870000.0,1,1.0,2024-12-27,5250.699707,2024.0,12.0,27.0,Hidro,NaN,NaN
2024-12-28,13870000.0,1,1.0,2024-12-28,5414.815430,2024.0,12.0,28.0,Hidro,NaN,NaN
2024-12-29,13870000.0,1,1.0,2024-12-29,5684.442383,2024.0,12.0,29.0,Hidro,NaN,NaN


In [80]:
stats

median          min          max
Data Data                                       
1    1     5948.250000  2287.580078  9478.429688
     2     6018.709961  2424.300049  9585.259766
     3     6132.299805  2546.629883  9674.750000
     4     6254.160156  2657.899902  9773.669922
     5     6399.080078  2733.419922  9873.089844
...                ...          ...          ...
12   27    5711.819824  1969.270020  8918.809570
     28    5711.819824  1965.239990  9039.759766
     29    5684.442383  1997.579956  9152.809570
     30    5767.020020  2079.610107  9266.559570
     31    5850.310059  2180.229980  9363.370117

[366 rows x 3 columns]

In [78]:
pd.date_range(start="2024-01-01", end="2026-12-31")

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10',
               ...
               '2026-12-22', '2026-12-23', '2026-12-24', '2026-12-25',
               '2026-12-26', '2026-12-27', '2026-12-28', '2026-12-29',
               '2026-12-30', '2026-12-31'],
              dtype='datetime64[ns]', length=1096, freq='D')